In [ ]:
# ==============================================================================
# UNAD - PROGRAMACIÓN
# FASE 4: COMPONENTE PRÁCTICO - PRÁCTICAS SIMULADAS
# DESARROLLO INDIVIDUAL - MANEJO DE EXCEPCIONES Y CONCEPTOS AVANZADOS POO
# ==============================================================================

import os

# ==============================================================================
# 1. DEFINICIÓN DE EXCEPCIONES PERSONALIZADAS
# ==============================================================================
class SaldoInsuficienteError(Exception):
    def __init__(self, saldo_actual, monto_solicitado):
        self.message = f"Error: Saldo insuficiente. Saldo disponible: ${saldo_actual}, Monto solicitado: ${monto_solicitado}"
        super().__init__(self.message)

class MontoInvalidoError(Exception):
    def __init__(self, monto):
        self.message = f"Error: El monto ingresado (${monto}) debe ser mayor a cero."
        super().__init__(self.message)

class AutenticacionError(Exception):
    def __init__(self):
        self.message = "Error: Autenticación fallida. El número de cuenta o PIN es incorrecto."
        super().__init__(self.message)


# Sistema de flujo de datos para auditoría local
def registrar_error_en_archivo(mensaje_error):
    try:
        with open("log_errores.txt", "a", encoding="utf-8") as archivo:
            archivo.write(f"[REGISTRO] {mensaje_error}\n")
    except IOError:
        print("No se pudo escribir en el archivo de registro de errores.")


# ==============================================================================
# 2. CLASE BASE (HERENCIA PRINCIPAL)
# ==============================================================================
class CuentaBancaria:
    def __init__(self, numero_cuenta, pin, titular, saldo_inicial=0.0):
        self._numero_cuenta = numero_cuenta  # Atributo protegido para herencia
        self._pin = pin                      # Atributo protegido
        self.titular = titular                # Atributo público
        self._saldo = float(saldo_inicial)   # Atributo protegido

    def verificar_credenciales(self, numero_cuenta, pin):
        return self._numero_cuenta == numero_cuenta and self._pin == pin

    def consultar_saldo(self):
        """Método base para consultar el saldo de la cuenta."""
        return self._saldo

    def depositar(self, monto):
        if monto <= 0:
            raise MontoInvalidoError(monto)
        self._saldo += monto
        print(f"✔️ Depósito exitoso. Nuevo saldo en su cuenta: ${self._saldo}")

    def retirar(self, monto):
        """Método abstracto que será sobreescrito por las clases hijas (Polimorfismo)."""
        raise NotImplementedError("Este método debe ser implementado por la subclase.")


# ==============================================================================
# 3. CLASES HIJAS (APLICANDO HERENCIA Y POLIMORFISMO)
# ==============================================================================

class CuentaAhorros(CuentaBancaria):
    """Clase hija que hereda de CuentaBancaria."""
    def __init__(self, numero_cuenta, pin, titular, saldo_inicial=0.0):
        super().__init__(numero_cuenta, pin, titular, saldo_inicial)
        self.__tasa_interes = 0.01  # Atributo exclusivo de la cuenta de ahorros

    # Polimorfismo: Sobreescritura del método consultar_saldo
    def consultar_saldo(self):
        print("(Consultando Cuenta de Ahorros)")
        return self._saldo

    # Polimorfismo: Implementación específica del método retirar para ahorros
    def retirar(self, monto):
        if monto <= 0:
            raise MontoInvalidoError(monto)
        if monto > self._saldo:
            raise SaldoInsuficienteError(self._saldo, monto)
        
        self._saldo -= monto
        print(f"✔️ Retiro exitoso de Cuenta de Ahorros. Saldo restante: ${self._saldo}")


class CuentaCorriente(CuentaBancaria):
    """Otra clase hija que hereda de CuentaBancaria."""
    def __init__(self, numero_cuenta, pin, titular, saldo_inicial=0.0, cupo_sobregiro=200000.0):
        super().__init__(numero_cuenta, pin, titular, saldo_inicial)
        self.__cupo_sobregiro = float(cupo_sobregiro)  # Atributo exclusivo de corriente

    # Polimorfismo: Sobreescritura del método consultar_saldo
    def consultar_saldo(self):
        print(f"(Consultando Cuenta Corriente - Cupo Sobregiro: ${self.__cupo_sobregiro})")
        return self._saldo

    # Polimorfismo: Implementación específica del método retirar para cuenta corriente
    def retirar(self, monto):
        if monto <= 0:
            raise MontoInvalidoError(monto)
        
        # Permite retirar dinero usando el cupo de sobregiro disponible
        limite_disponible = self._saldo + self.__cupo_sobregiro
        if monto > limite_disponible:
            raise SaldoInsuficienteError(limite_disponible, monto)
        
        self._saldo -= monto
        print(f"✔️ Retiro exitoso de Cuenta Corriente (Con sobregiro). Saldo actual: ${self._saldo}")


# ==============================================================================
# 4. INTERFAZ DE USUARIO Y CONTROL DE FLUJO
# ==============================================================================
def ejecutar_cajero():
    # Creamos una cuenta de ahorros para pruebas
    cuenta_usuario = CuentaAhorros(numero_cuenta="123456", pin="9876", titular="Félix Dávila", saldo_inicial=500000.0)
    
    print("=== BIENVENIDO AL SISTEMA DE CAJERO AUTOMÁTICO OPTIMIZADO (UNAD) ===")
    
    intentos = 3
    autenticado = False
    
    while intentos > 0 and not autenticado:
        try:
            print(f"\n[Intentos restantes: {intentos}]")
            u_cuenta = input("Ingrese su número de cuenta: ").strip()
            u_pin = input("Ingrese su PIN de seguridad: ").strip()
            
            if not cuenta_usuario.verificar_credenciales(u_cuenta, u_pin):
                raise AutenticacionError()
            
            autenticado = True
            print(f"\n🎉 ¡Bienvenido(a) {cuenta_usuario.titular}!")
            
        except AutenticacionError as e:
            print(e)
            registrar_error_en_archivo(e.message)
            intentos -= 1
            if intentos == 0:
                print("❌ Cuenta bloqueada por seguridad tras múltiples intentos fallidos.")
                registrar_error_en_archivo("Cuenta bloqueada por exceso de intentos.")
                return

    while autenticado:
        print("\n--- MENÚ DE OPCIONES ---")
        print("1. Consultar Saldo")
        print("2. Realizar Depósito")
        print("3. Realizar Retiro")
        print("4. Salir")
        
        try:
            opcion = input("Seleccione una opción (1-4): ").strip()
            
            if opcion == "1":
                # Aquí se ejecuta el Polimorfismo de forma interna
                print(f"💰 Su saldo disponible es: ${cuenta_usuario.consultar_saldo()}")
                
            elif opcion == "2":
                try:
                    monto = float(input("Ingrese el monto a depositar: $"))
                    cuenta_usuario.depositar(monto)
                except ValueError:
                    msg = "Error de Entrada: Se intentó ingresar texto en depósito."
                    print(msg)
                    registrar_error_en_archivo(msg)
                except MontoInvalidoError as e:
                    print(e)
                    registrar_error_en_archivo(e.message)
                    
            elif opcion == "3":
                try:
                    monto = float(input("Ingrese el monto a retirar: $"))
                    # Aquí se ejecuta el Polimorfismo para retirar según el tipo de cuenta
                    cuenta_usuario.retirar(monto)
                except ValueError:
                    msg = "Error de Entrada: Se intentó ingresar texto en retiro."
                    print(msg)
                    registrar_error_en_archivo(msg)
                except (MontoInvalidoError, SaldoInsuficienteError) as e:
                    print(e)
                    registrar_error_en_archivo(e.message)
                    
            elif opcion == "4":
                print("👋 Gracias por utilizar nuestros servicios virtuales. ¡Hasta pronto!")
                break
            else:
                print("⚠ Opción no válida. Intente de nuevo.")
                
        except Exception as e:
            msg_inesperado = f"Error imprevisto en el sistema: {e}"
            print(msg_inesperado)
            registrar_error_en_archivo(msg_inesperado)

if __name__ == "__main__":
    ejecutar_cajero()